In [1]:
!pip install ipython-sql
!pip install pyodbc

In [2]:
import pandas as pd
import pyodbc

In [4]:
conn = pyodbc.connect("Driver={SQL Server};Server=localhost\\SQLEXPRESS;Database=cw2;")


In [5]:
conn.close()

In [7]:
conn = pyodbc.connect("Driver={SQL Server};Server=localhost\\SQLEXPRESS;Database=cw2;")

In [8]:
cursor= conn.cursor()

In [11]:
conn.close()

In [12]:
conn = pyodbc.connect("Driver={SQL Server};Server=localhost\\SQLEXPRESS;Database=cw2;")
cursor = conn.cursor()

insert_car_query = """
INSERT INTO car (regnum, make, model, dateofman, mileage, nextsdate)
VALUES (?, ?, ?, ?, ?, ?);
"""
car_values = ['AAA 123', 'Volkswagen', 'Polo', '2024-06-30', 49904, '2024-09-18']

cursor.execute(insert_car_query, car_values)

insert_service_query = """
INSERT INTO service (sid, dropoff_date, dropoff_time, worktext, regnum)
VALUES (?, ?, ?, ?, ?);
"""
service_values = ['S2006-136', '2024-09-12', '14:30:00', 'New service has been booked', 'AAA 123']
cursor.execute(insert_service_query, service_values)

conn.commit()
conn.close()


In [18]:
conn = pyodbc.connect("Driver={SQL Server};Server=localhost\\SQLEXPRESS;Database=cw2;")
cursor = conn.cursor()

###############################
selectQuery = """
SELECT e.empname AS mechanic_name, e.grade AS mechanic_grade, COUNT(w.[sid]) AS number_of_jobs 
FROM work w
JOIN emp e ON w.empid = e.empid
JOIN [service] s1 ON w.[sid] = s1.[sid]
JOIN [service] s2 ON s1.dropoff_date = s2.dropoff_date AND s1.regnum = s2.regnum
WHERE s2.regnum = 'CEZ 563'
GROUP BY e.empname, e.grade 
HAVING COUNT(w.[sid]) > 0
ORDER BY number_of_jobs DESC;
"""

result = cursor.execute(selectQuery)
resultList = []
for row in result:
    resultList.append(list(row))  # Convert each row to a list

tableColumns = ["Mechanic Name", "Mechanic Grade", "Number of Jobs"]
resultTable = pd.DataFrame(resultList, columns=tableColumns)
display(resultTable)
conn.close()

,Mechanic Name,Mechanic Grade,Number of Jobs
0,Tony Hoare,Apprentice,1


In [20]:
conn = pyodbc.connect("Driver={SQL Server};Server=localhost\\SQLEXPRESS;Database=cw2;")
cursor = conn.cursor()

###############################
sql_query = """
DECLARE @start_date DATE = '2020-06-22';
DECLARE @end_date DATE = '2020-06-22';
SELECT emp.empid, emp.empname, 
       SUM(DATEDIFF(SECOND, '00:00:00', work.timespent)) / 60.0 AS total_minutes_spent
FROM work, emp
WHERE work.sid IN 
    (SELECT service.sid 
    FROM service 
    WHERE service.dropoff_date BETWEEN @start_date AND @end_date) 
AND work.empid IN (SELECT empid FROM emp)
GROUP BY emp.empid, emp.empname
ORDER BY total_minutes_spent DESC;
"""

result = cursor.execute(sql_query)
resultList = []
for row in result:
    resultList.append(list(row)) 
    
tableColumns = ["Employee ID", "Employee Name", "Total Minutes Spent"]
resultTable = pd.DataFrame(resultList, columns=tableColumns)
display(resultTable)
conn.close()


,Employee ID,Employee Name,Total Minutes Spent
0,E0392,Edsger Dijkstra,565.000000
1,E1037,Edgar F. Codd,565.000000
2,E2045,Grace Hopper,565.000000
3,E2648,Alan Turing,565.000000
4,E4470,Ada Lovelace,565.000000
5,E7291,Tony Hoare,565.000000
6,E9274,Tim Berners-Lee,565.000000
